In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

class GrayScottSimulator(torch.nn.Module):
    def __init__(self, Du=0.16, Dv=0.08, F_rate=0.04, k=0.06, dx=1.0, dt=1.0):
        """
        Default parameters (F=0.04, k=0.06) correspond to the famous chaotic 
        "coral" or "mitosis" Turing patterns, perfect for UQ benchmarking.
        """
        super().__init__()
        self.Du = Du
        self.Dv = Dv
        self.F = F_rate
        self.k = k
        self.dx = dx
        self.dt = dt

        # 5-point spatial Laplacian stencil
        laplacian_kernel = torch.tensor([
            [0.0,  1.0, 0.0],
            [1.0, -4.0, 1.0],
            [0.0,  1.0, 0.0]
        ]) / (dx ** 2)
        
        # Reshape for depthwise conv2d: (out_channels, in_channels/groups, H, W)
        # We repeat it twice so we can compute the Laplacian for U and V simultaneously
        self.register_buffer('kernel', laplacian_kernel.view(1, 1, 3, 3).repeat(2, 1, 1, 1))

    def step(self, uv):
        """Performs a single Forward Euler time step."""
        u = uv[:, 0:1, :, :]
        v = uv[:, 1:2, :, :]

        # Apply periodic boundary conditions using circular padding
        uv_padded = F.pad(uv, (1, 1, 1, 1), mode='circular')
        
        # Compute Laplacian for both U and V simultaneously using grouped convolution
        laplacian = F.conv2d(uv_padded, self.kernel, groups=2)
        lap_u = laplacian[:, 0:1, :, :]
        lap_v = laplacian[:, 1:2, :, :]

        # Core Reaction-Diffusion dynamics
        uvv = u * v * v
        du = self.Du * lap_u - uvv + self.F * (1.0 - u)
        dv = self.Dv * lap_v + uvv - (self.F + self.k) * v

        # Euler update
        u_next = u + self.dt * du
        v_next = v + self.dt * dv

        return torch.cat([u_next, v_next], dim=1)

    def generate_trajectory(self, batch_size=4, size=64, steps=50, sub_steps=20, device='cuda'):
        """
        Generates a batched sequence of data over time.
        'sub_steps' is the number of internal PDE solver steps between each saved ML frame.
        """
        # Initialize background: U = 1, V = 0
        uv = torch.zeros((batch_size, 2, size, size), device=device)
        uv[:, 0, :, :] = 1.0

        # Seed the simulation with random localized perturbations
        for b in range(batch_size):
            for _ in range(torch.randint(2, 5, (1,)).item()): # 2 to 4 random seeds per batch
                x, y = torch.randint(5, size - 15, (2,))
                noise_u = 0.50 + 0.1 * torch.rand(10, 10, device=device)
                noise_v = 0.25 + 0.1 * torch.rand(10, 10, device=device)
                uv[b, 0, x:x+10, y:y+10] = noise_u
                uv[b, 1, x:x+10, y:y+10] = noise_v

        trajectory = [uv.clone()]
        
        # Rollout the simulation
        for _ in range(steps):
            for _ in range(sub_steps): # Take multiple small solver steps for stability
                uv = self.step(uv)
            trajectory.append(uv.clone())
            
        # Returns shape: (Time, Batch, Channels, Height, Width)
        return torch.stack(trajectory)



In [ ]:

device = 'cuda' if torch.cuda.is_available() else 'cpu'
simulator = GrayScottSimulator().to(device)

print("Generating 50 timesteps of Gray-Scott data...")
# Generate a batch of 4 trajectories, 64x64 resolution, 50 ML timesteps
data = simulator.generate_trajectory(batch_size=4, size=64, steps=50, sub_steps=40, device=device)

print(f"Data shape: {data.shape} -> (Time, Batch, Channels, Height, Width)")

# Plot the final state (V channel) of the first batch item
plt.imshow(data[-1, 0, 1, :, :].cpu().numpy(), cmap='magma')
plt.title("Gray-Scott Pattern (V Channel) at Final Step")
plt.colorbar()
plt.show()